In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_tissue_artifact.pkl"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr" 

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import with_tissue_artifact

artifacts = True
if artifacts: 
    df_sub = with_tissue_artifact(df_all, cache_file, segmentation_type = "artifact", status="complete", version="default")
    all_filenames = df_sub["filename"].tolist()
    print("Number of slides with artifact segmentation performed: ", len(all_filenames))

In [ ]:
# Feature Results
feature_result = r"D:\NOTEBOOKS\Christine\all_slides\feature_summary_new.csv"
df_feature_result = pd.read_csv(feature_result)
df_feature_result = df_feature_result[df_feature_result['status'] == "feature extraction complete"].copy()
df_features = df_feature_result[df_feature_result['model'] == "h-optimus-0"].copy()
with_features = set(df_features["wsi_path"])
print(len(with_features))

df_sub = df_sub[df_sub["filename"].isin(with_features)]
print(len(df_sub))

In [ ]:
all_filenames = df_sub["filename"].tolist()
print("Number of wsi filenames: ", len(all_filenames))

In [ ]:
print(df_all)

In [ ]:
# UMAP of tile-level features
import os
from wsidata import open_wsi
import scanpy as sc
import pandas as pd
import geopandas as gpd

adatas = []

for i, path in enumerate(all_filenames):
    zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))

    wsi = open_wsi(path, zarr_path)
    adata = wsi.tables["features_h-optimus-0"]
    tiles = wsi.shapes["tiles_224"]
    artifacts = wsi.shapes["artifacts_grandqc"]

    # Keep only needed columns
    tiles = tiles[["tile_id", "geometry"]]
    artifacts = artifacts[["class", "geometry"]]

    # Spatial join: assign artifact class to tiles that overlap artifact polygons
    joined = gpd.sjoin(
        tiles,
        artifacts,
        how="left",
        predicate="intersects"
    )

    # If multiple artifacts overlap one tile, keep the first one
    joined = (
        joined.groupby("tile_id")["class"]
        .first()
        .reset_index()
    )

    # Merge artifact labels into adata.obs
    adata.obs = adata.obs.merge(
        joined,
        on="tile_id",
        how="left"
    )

    # Fill tiles without artifacts
    adata.obs["artifact_type"] = (
        adata.obs["class"]
        .fillna("No artifact")
        .astype(str)
    )

    # Optional cleanup
    adata.obs.drop(columns=["class"], inplace=True)

    adatas.append(adata)

# concatenate all slides
adata_concat = sc.concat(
    adatas,
    label="slide_id",
    keys=[f"slide_{i}" for i in range(len(adatas))]
)

adata_concat.obs["artifact_type"] = pd.Categorical(
    adata_concat.obs["artifact_type"]
)

In [ ]:
sc.pp.scale(adata_concat)
sc.pp.pca(adata_concat)
sc.pp.neighbors(adata_concat)
sc.tl.umap(adata_concat)

In [ ]:
sc.pl.umap(adata_concat, color='artifact_type', size=3)